In [ ]:
# Parsing

from html.parser import HTMLParser
import numpy as np

class TableParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.in_tbody = False
        self.in_td = False
        self.rows = []
        self.current_row = []
        self.current_data = ''

    def handle_starttag(self, tag, attrs):
        if tag == 'tbody': self.in_tbody = True
        if tag == 'td' and self.in_tbody: self.in_td = True; self.current_data = ''

    def handle_endtag(self, tag):
        if tag == 'tbody': self.in_tbody = False
        if tag == 'td' and self.in_td:
            self.in_td = False
            self.current_row.append(self.current_data.strip())
        if tag == 'tr' and self.in_tbody and self.current_row:
            self.rows.append(self.current_row)
            self.current_row = []

    def handle_data(self, data):
        if self.in_td: self.current_data += data

with open('ssa_life_table_2022.html') as f:
    html = f.read()

parser = TableParser()
parser.feed(html)

rows = parser.rows[:-1]

# Probability of dying within one year, male/female
death_prob_m = {int(l[0]): float(l[1]) for l in rows}
death_prob_f = {int(l[0]): float(l[4]) for l in rows}

In [ ]:
import matplotlib.pyplot as plt


def build_table(age_from, age_to, step):
    table = {}
    for gender, death_prob in [("m", death_prob_m), ("f", death_prob_f)]:
        for current_age in range(age_from, age_to + 1, step):
            for death_age in range(current_age, age_to + 1, step):
                not_die = 1
                for age in range(current_age, death_age + 1):
                    not_die *= (1 - death_prob[age])
                die = 1 - not_die
                table[(gender, current_age, death_age)] = f"1/{1/die:.0f}" if die < 0.1 else f"{die:.2f}"
    return table


def render_and_save(table, age_from, age_to, step, filename):
    EDGE = 0.7
    BORDER_COLOR = '#999999'
    BORDER_WIDTH = 1.0
    COLORS = {'m': '#dce9f7', 'f': '#fce4ec'}

    ages = list(range(age_from, age_to + 1, step))
    n = len(ages)

    # Scale figure and fonts to grid size
    fig_size = max(14, n * 0.85)
    cell_px = fig_size / (n + 2 * EDGE)
    cell_font = min(16, cell_px * 18)
    diag_font = cell_font * 0.75
    label_font = min(18, cell_px * 20)

    fig, ax = plt.subplots(figsize=(fig_size, fig_size))

    for (gender, ca, da), label in table.items():
        ci = (ca - age_from) // step
        di = (da - age_from) // step
        if ci == di:
            continue
        if gender == 'm':
            x, y = EDGE + di, EDGE + ci
        else:
            x, y = EDGE + ci, EDGE + di
        ax.add_patch(plt.Rectangle((x, y), 1, 1, facecolor=COLORS[gender],
                                   edgecolor=BORDER_COLOR, linewidth=BORDER_WIDTH))
        ax.text(x + 0.5, y + 0.5, label, ha='center', va='center',
                fontsize=cell_font, color='black', weight='bold')

    for i, age in enumerate(ages):
        x, y = EDGE + i, EDGE + i
        m_label = table.get(('m', age, age), '')
        f_label = table.get(('f', age, age), '')
        ax.add_patch(plt.Rectangle((x, y), 1, 0.5, facecolor=COLORS['m'],
                                   edgecolor=BORDER_COLOR, linewidth=BORDER_WIDTH))
        ax.text(x + 0.5, y + 0.25, m_label, ha='center', va='center',
                fontsize=diag_font, color='black', weight='bold')
        ax.add_patch(plt.Rectangle((x, y + 0.5), 1, 0.5, facecolor=COLORS['f'],
                                   edgecolor=BORDER_COLOR, linewidth=BORDER_WIDTH))
        ax.text(x + 0.5, y + 0.75, f_label, ha='center', va='center',
                fontsize=diag_font, color='black', weight='bold')

    for i, age in enumerate(ages):
        c = EDGE + i + 0.5
        lbl = str(age)
        ax.text(c, EDGE * 0.5, lbl, ha='center', va='center', fontsize=label_font, weight='bold')
        ax.text(c, EDGE + n + EDGE * 0.5, lbl, ha='center', va='center', fontsize=label_font, weight='bold')
        ax.text(EDGE * 0.5, c, lbl, ha='center', va='center', fontsize=label_font, weight='bold')
        ax.text(EDGE + n + EDGE * 0.5, c, lbl, ha='center', va='center', fontsize=label_font, weight='bold')

    total = EDGE + n + EDGE
    ax.set_xlim(0, total)
    ax.set_ylim(total, 0)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title("Death probability from age N to age M, incl (Data: SSA, U.S. Gov)",
                 fontsize=20, weight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()


def generate_mortality_chart(age_from, age_to, step, filename):
    table = build_table(age_from, age_to, step)
    render_and_save(table, age_from, age_to, step, filename)

In [ ]:
generate_mortality_chart(20, 100, 5, "mortality-20-100.png")
generate_mortality_chart(1, 116, 5, "mortality-1-116.png")
generate_mortality_chart(2, 117, 5, "mortality-2-117.png")
generate_mortality_chart(3, 118, 5, "mortality-3-118.png")
generate_mortality_chart(4, 119, 5, "mortality-4-119.png")
generate_mortality_chart(5, 115, 5, "mortality-5-115.png")